# Reference Trajectory Analysis
Plot planner-generated reference trajectories (`joints`, `joints_target`, `joint_vel`)
and compare upsampling strategies (linear interpolation vs first-order IIR).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import interp1d

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

joint_names = ["shoulder_pan", "shoulder_lift", "elbow", "wrist_1", "wrist_2", "wrist_3"]

In [ ]:
def upsample_linear(x, k):
    N = x.shape[0]
    t_old = np.arange(N)
    t_new = np.linspace(0, N - 1, k * (N - 1) + 1)
    f = interp1d(t_old, x, axis=0, kind="linear")
    return f(t_new)


def _load_reference(traj_path):
    traj_path = Path(traj_path)
    traj = np.load(traj_path)
    joints = traj["joints"] if "joints" in traj.files else traj["joints_l"]
    joints_target = traj["joints_target"] if "joints_target" in traj.files else traj["joints_target_l"]
    joint_vel = traj["joint_vel"] if "joint_vel" in traj.files else traj["joint_vel_l"]
    dt = float(traj["dt"]) if "dt" in traj.files else 0.02

    return traj_path, joints, joints_target, joint_vel, dt

In [ ]:
def plot_reference(traj_path, ds=1, start_i=0):
    """Plot joint_pos and joint_target on the left axis, joint_vel on a twin right axis."""
    traj_path, joints, joints_target, joint_vel, dt = _load_reference(traj_path)
    joints = joints[start_i:]
    joints_target = joints_target[start_i:]
    joint_vel = joint_vel[start_i:]
    time = np.arange(len(joints)) * dt

    fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    for i, (ax, name) in enumerate(zip(axes.flat, joint_names)):
        ax.plot(time[::ds], joints[::ds, i], color='tab:blue', alpha=0.9, label='joints')
        ax.plot(time[::ds], joints_target[::ds, i], color='tab:orange', linestyle='--', alpha=0.85,
                label='joints_target')
        ax.set_title(f"joint {i}: {name}", fontsize=10)
        ax.set_ylabel('position (rad)')
        ax.grid(True, alpha=0.3)

        ax_v = ax.twinx()
        ax_v.plot(time[::ds], joint_vel[::ds, i], color='tab:green', alpha=0.6, linewidth=1,
                  label='joint_vel')
        ax_v.axhline(0, color='tab:green', linewidth=0.3, alpha=0.3)
        ax_v.set_ylabel('velocity (rad/s)', color='tab:green')
        ax_v.tick_params(axis='y', labelcolor='tab:green')

        if i == 0:
            lines_l, labels_l = ax.get_legend_handles_labels()
            lines_r, labels_r = ax_v.get_legend_handles_labels()
            ax.legend(lines_l + lines_r, labels_l + labels_r, fontsize=7, loc='best')
    axes[-1, 0].set_xlabel('time (s)')
    axes[-1, 1].set_xlabel('time (s)')
    fig.suptitle(f"reference trajectory — {traj_path.stem}", fontsize=11)
    plt.tight_layout()

In [ ]:
def analyze_reference(traj_path, alpha=0.1, ds=5):
    """Plot linear-interpolation vs first-order IIR smoothing of the planner's joint targets.

    Both methods upsample the 50 Hz planner waypoints to 500 Hz (decimation=10):
      - linear: piecewise-linear between successive targets (current control-thread behavior)
      - IIR:    y[n] = (1 - alpha) * y[n-1] + alpha * u[n], with u[n] a ZOH of the targets.
                alpha in (0, 1]; smaller = smoother / more lag. alpha=1 collapses to ZOH.
    """
    traj_path, joints, joints_target, _, _ = _load_reference(traj_path)

    joints_target = np.vstack([joints[0:1], joints_target])           # (T+1, 6)

    decimation = 10  # 50 Hz planner -> 500 Hz control

    # Shared command waveform: starts at joints[0], switches to joints_target[i] at step (i+1)*decimation.
    traj_x = np.arange(len(joints_target)) * decimation               # 0, 10, 20, ...
    ref_steps = np.arange(traj_x[-1] + 1)                             # 0 .. T*decimation

    # --- Linear interpolation ---
    linear = upsample_linear(joints_target, decimation)

    # --- First-order IIR low-pass on the ZOH-upsampled command ---
    iir = np.empty_like(linear)
    iir[0] = joints[0]
    one_minus_alpha = 1.0 - alpha
    for i, target in enumerate(joints_target[:-1]):
        for j in range(decimation):
            n = i * decimation + j
            iir[n + 1] = one_minus_alpha * iir[n] + alpha * target

    # --- Plot ---
    fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    for i, (ax, name) in enumerate(zip(axes.flat, joint_names)):
        ax.plot(traj_x, joints_target[:, i], 'o', markersize=2.5, color='k',
                alpha=0.5, label='planner waypoints')
        ax.plot(ref_steps[::ds], linear[::ds, i], label='linear interp',
                color='tab:green', alpha=0.85)
        ax.plot(ref_steps[::ds], iir[::ds, i], label=f'IIR (\u03b1={alpha})',
                color='tab:red', alpha=0.85)
        ax.set_title(f"joint {i}: {name}", fontsize=10)
        ax.set_ylabel('position (rad)')
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=8, loc='best')
    axes[-1, 0].set_xlabel('step (500Hz)')
    axes[-1, 1].set_xlabel('step (500Hz)')
    fig.suptitle(f"reference joint targets — {traj_path.stem}", fontsize=11)
    plt.tight_layout()

In [ ]:
plot_reference("../reference_trajectories/box_push_ur5e/traj_full_refined_20260413_163929_cubic.npz")

In [ ]:
analyze_reference("../reference_trajectories/box_push_ur5e/traj_full_refined_20260413_163929_cubic.npz", ds=5, alpha=0.05)

In [ ]:
traj_files = [
    "../reference_trajectories/box_push_ur5e/traj_full_20260413_163929_cubic.npz",
    "../reference_trajectories/box_push_ur5e/traj_full_20260413_163929_linear.npz",
    "../reference_trajectories/box_push_ur5e/traj_full_refined_20260413_163929_linear.npz",
    "../reference_trajectories/box_push_ur5e/traj_full_refined_20260413_163929_cubic.npz",
]

start_i = 300

for j in range(6):
    fig, ax = plt.subplots(figsize=(14, 7))
    for path in traj_files:
        traj = np.load(path)
        jt = traj["joints_target"] if "joints_target" in traj.files else traj["joints_target_l"]
        ax.plot(jt[start_i:, j], label=Path(path).stem)
    ax.set_xlabel("timestep (50 Hz)")
    ax.set_ylabel("position (rad)")
    ax.set_title(f"joint {j}: {joint_names[j]} — targets across trajectories")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

In [ ]:
traj_files = [
    "../reference_trajectories/box_push_ur5e/traj_full_refined_20260413_163929_cubic.npz",
]


alpha = 0.03
decimation = 10
start_i = 0 * decimation
end_i = start_i + decimation * 1000

for joint_idx in range(6):
    fig, ax = plt.subplots(figsize=(14, 5))
    for path in traj_files:
        traj = np.load(path)
        jt = traj["joints_target"] if "joints_target" in traj.files else traj["joints_target_l"]
        joints = traj["joints"] if "joints" in traj.files else traj["joints_l"]
        jt = np.vstack([joints[0:1], jt])

        linear = upsample_linear(jt, decimation)

        iir = np.empty_like(linear)
        iir[0] = joints[0]
        for i, target in enumerate(jt[:-1]):
            for j in range(decimation):
                n = i * decimation + j
                iir[n + 1] = (1 - alpha) * iir[n] + alpha * target

        label = Path(path).stem
        color = ax._get_lines.get_next_color()
        ax.plot(linear[start_i:(end_i+1), joint_idx], color=color, linestyle=':', alpha=0.85,
                label=f'{label} (linear)')
        ax.plot(iir[start_i:(end_i+1), joint_idx], color=color, linestyle='-', alpha=0.85,
                label=f'{label} (IIR \u03b1={alpha})')

    ax.set_xlabel("step (500 Hz)")
    ax.set_ylabel("position (rad)")
    ax.set_title(f"joint {joint_idx}: {joint_names[joint_idx]} — linear vs IIR (\u03b1={alpha})")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

In [ ]:
plot_reference("../reference_trajectories/box_rotate_ur5e/traj_full_refined_20260417_134041_cubic.npz")

In [ ]:
data = np.load("../reference_trajectories/box_rotate_ur5e/traj_full_refined_20260417_134041_cubic.npz")

obj_vel = data["obj_vel"]
obj_vel_lin = np.linalg.norm(obj_vel[:,:3], axis=1)
obj_vel_ang = np.linalg.norm(obj_vel[:,3:], axis=1)

eps = 1e-3

k = 25

in_contact = (obj_vel_lin + obj_vel_ang) > eps
near_contact = np.copy(in_contact)
for i, is_moving in enumerate(in_contact):
    if is_moving:
        for j in range(i+k, i-k-1, -1):
            if j >= 0 and j < len(near_contact):
                near_contact[j] = True

plt.plot(near_contact)

print(np.where(near_contact)[0].shape)

In [ ]:
def _load_rollout(rollout_path):
    """Load a rollout NPZ produced by scripts/rsl_rl/record.py and return a dict
    matching the legacy JSON layout used by these analysis functions."""
    npz = np.load(rollout_path)
    rollout = {
        "joint_positions": npz["actual_q"],   # (N, n_joints)
        "joint_targets": npz["target_q"],     # (N, n_joints)
        "trajectory_path": str(npz["trajectory_path"]),
        "action_scale": (
            float(npz["action_scale"]) if npz["action_scale"].ndim == 0
            else npz["action_scale"].tolist()
        ),
        "action_mode": str(npz["action_mode"]),
        "dual_arm": bool(npz["dual_arm"]),
    }
    return rollout


def plot_action_norm_vs_obj_vel(rollout_npz_path, ds=1):
    """Overlay per-step L2 norm of the policy's residual action with the reference box velocity.

    Residual is recovered from `joint_targets` in the rollout, inverting the action_mode
    transformation saved on the run. record.py starts rollouts at trajectory index 0
    (`_reset_idx(None, 0)`), so rollout step t maps directly to traj index t.
    """
    rollout_npz_path = Path(rollout_npz_path)
    rollout = _load_rollout(rollout_npz_path)

    joint_targets   = np.array(rollout["joint_targets"])                 # (N, D)
    joint_positions = np.array(rollout["joint_positions"])               # (N, D)
    action_scale = float(rollout["action_scale"])
    action_mode  = rollout.get("action_mode", "A")
    dual_arm     = bool(rollout.get("dual_arm", False))

    # Resolve trajectory path relative to repo root.
    # rollout NPZ lives at <repo>/logs/rsl_rl/<task>/<run>/rollout/output.npz → .parents[5] = repo.
    traj_path = Path(rollout["trajectory_path"])
    if not traj_path.is_absolute():
        traj_path = rollout_npz_path.parents[5] / traj_path
    traj = np.load(traj_path)

    if dual_arm:
        ref_target = np.concatenate([traj["joints_target_l"], traj["joints_target_r"]], axis=-1)
        ref_joints = np.concatenate([traj["joints_l"],        traj["joints_r"]],        axis=-1)
    else:
        ref_target = traj["joints_target"]
        ref_joints = traj["joints"]

    obj_vel     = traj["obj_vel"]
    obj_vel_lin = np.linalg.norm(obj_vel[:, :3], axis=1)
    obj_vel_ang = np.linalg.norm(obj_vel[:, 3:], axis=1)
    dt = float(traj["dt"]) if "dt" in traj.files else 0.02

    N = len(joint_targets)

    # Invert the action_mode transform to recover the raw policy action.
    if action_mode == "A":
        actions = (joint_targets - ref_target[:N]) / action_scale
    elif action_mode == "B":
        actions = (joint_targets - ref_joints[:N]) / action_scale
    elif action_mode == "C":
        actions = (joint_targets - joint_positions[:N]) / action_scale
    elif action_mode == "D":
        ff = (ref_target - ref_joints)[:N]
        actions = (joint_targets - joint_positions[:N] - ff) / action_scale
    else:
        raise ValueError(f"Unknown action_mode: {action_mode!r}")

    action_norm = np.linalg.norm(actions, axis=-1)

    time_rollout = np.arange(N) * dt
    time_ref     = np.arange(len(obj_vel_lin)) * dt

    fig, ax = plt.subplots(figsize=(14, 5))
    ax2 = ax.twinx()
    ax2.fill_between(time_ref, 0, obj_vel_lin, color="tab:blue", alpha=0.18, label="|obj v_lin| (m/s)")
    ax2.fill_between(time_ref, 0, obj_vel_ang, color="tab:red",  alpha=0.18, label="|obj v_ang| (rad/s)")
    ax2.set_ylabel("box velocity")

    label = "||action||"
    ax.plot(time_rollout[::ds], action_norm[::ds], color="tab:green", linewidth=1.5, label=label)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("action L2 norm")
    ax.grid(True, alpha=0.3)

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=9, loc="best")

    run_name = rollout_npz_path.parent.parent.name
    ax.set_title(
        f"action norm vs box velocity — {run_name}  (mode={action_mode}, scale={action_scale})",
        fontsize=11,
    )
    plt.tight_layout()


def plot_target_diff(rollout_npz_path, ds=1):
    """Overlay per-step L2 norm of the policy's residual action with the reference box velocity.

    Residual is recovered from `joint_targets` in the rollout, inverting the action_mode
    transformation saved on the run. record.py starts rollouts at trajectory index 0
    (`_reset_idx(None, 0)`), so rollout step t maps directly to traj index t.
    """
    rollout_npz_path = Path(rollout_npz_path)
    rollout = _load_rollout(rollout_npz_path)

    joint_targets   = np.array(rollout["joint_targets"])                 # (N, D)
    joint_positions = np.array(rollout["joint_positions"])               # (N, D)
    action_scale = float(rollout["action_scale"])
    action_mode  = rollout.get("action_mode", "A")
    dual_arm     = bool(rollout.get("dual_arm", False))

    # Resolve trajectory path relative to repo root.
    traj_path = Path(rollout["trajectory_path"])
    if not traj_path.is_absolute():
        traj_path = rollout_npz_path.parents[5] / traj_path
    traj = np.load(traj_path)

    obj_vel     = traj["obj_vel"]
    dt = float(traj["dt"]) if "dt" in traj.files else 0.02

    N = len(joint_targets)

    target_diff = joint_targets - joint_positions[:N]

    time_rollout = np.arange(N) * dt

    fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    for i, (ax, name) in enumerate(zip(axes.flat, joint_names)):
        ax.plot(time_rollout[::ds], target_diff[::ds, i], color="tab:green", linewidth=1.5, label=label)
        ax.set_xlabel("time (s)")
        ax.set_ylabel("q_target - q_pos")
        ax.grid(True, alpha=0.3)

    h1, l1 = ax.get_legend_handles_labels()
    ax.legend(h1, l1, fontsize=9, loc="best")

    run_name = rollout_npz_path.parent.parent.name
    ax.set_title(
        f"joint target diff — {run_name}  (mode={action_mode}, scale={action_scale})",
        fontsize=11,
    )
    plt.tight_layout()

In [ ]:
plot_action_norm_vs_obj_vel(
    "../logs/rsl_rl/boxpush/2026-04-22_12-21-06/rollout/output.npz",
    ds=1,
)

In [ ]:
data = np.load("../reference_trajectories/box_rotate_ur5e/traj_full_refined_20260417_134041_cubic.npz")

q_pos = data["joints"]
q_target = data["joints_target"]
obj_vel = data["obj_vel"]

q_diff = q_target - q_pos

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
for i, (ax, name) in enumerate(zip(axes.flat, joint_names)):
    ax.plot(obj_vel[:, i])
    # ax.plot(q_pos[250:300,i])
    # ax.plot(q_target[250:300,i])

In [ ]:
print(q_pos[265] - q_target[265])

In [ ]:
plot_target_diff(
    "../logs/rsl_rl/boxpush/2026-04-22_12-21-06/rollout/output.npz",
    ds=1,
)